In [ ]:
# =============================================================================
# Experiment 4: Malicious URL Classification using Logistic Regression
# Objective: Build a logistic regression model to classify URLs as either legitimate or malicious. [CO2]
# =============================================================================

# 1. Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from urllib.parse import urlparse
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, roc_curve

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# =============================================================================
# 2. Create / Load URL Dataset (Legitimate vs. Malicious)
# Label: 0 = Legitimate, 1 = Malicious
# =============================================================================

legitimate_urls = [
    'https://www.google.com', 'https://www.github.com/explore', 'https://www.wikipedia.org/wiki/Python',
    'https://www.amazon.in/gp/bestsellers', 'https://docs.python.org/3/library/', 'https://stackoverflow.com/questions',
    'https://www.microsoft.com/en-in', 'https://www.linkedin.com/feed/', 'https://www.jainuniversity.ac.in/',
    'https://www.coursera.org/browse', 'https://www.nytimes.com/section/technology', 'https://www.bbc.com/news',
    'https://www.apple.com/in/mac/', 'https://www.youtube.com/watch?v=dQw4w9WgXcQ', 'https://www.kaggle.com/datasets',
    'https://pandas.pydata.org/docs/', 'https://scikit-learn.org/stable/', 'https://www.reddit.com/r/cybersecurity/',
    'https://www.cloudflare.com/learning/', 'https://www.mit.edu/research/'
] * 25

malicious_urls = [
    'http://192.168.1.45/login/verify-account.php', 'http://paypal-security-update-login.com/auth',
    'http://secure-bankofamerica-update.xyz/signin?id=99281', 'http://free-gift-card-claim-now.tk/win',
    'http://appleid-verify-support.net/account/confirm', 'http://amazon-order-refund-991.info/billing',
    'http://104.21.45.88/malware/payload.exe', 'http://login-microsoftonline-auth.ru/owa/auth',
    'http://netflix-payment-declined-update.club/card', 'http://bit.ly/3xYzMaliciousRedirect99',
    'http://verify-your-wallet-metamask.io.phish.co/seed', 'http://admin-panel-access-portal.biz/wp-login.php',
    'http://google-drive-shared-doc-2026.gq/download.zip', 'http://sbi-kyc-update-alert.in.net/otp',
    'http://203.0.113.19/shell.sh', 'http://instagram-blue-badge-apply.cf/login',
    'http://hdfcbank-netbanking-verify.top/user', 'http://dropbox-file-invoice-9821.ml/doc',
    'http://steam-free-skins-giveaway.xyz/trade', 'http://icici-reward-points-redeem.pw/claim'
] * 25

urls = legitimate_urls + malicious_urls
labels = [0] * len(legitimate_urls) + [1] * len(malicious_urls)

df = pd.DataFrame({'URL': urls, 'Label': labels})
# Shuffle dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total URLs: {len(df)} | Legitimate (0): {(df['Label']==0).sum()} | Malicious (1): {(df['Label']==1).sum()}")
print(df.head())

In [ ]:
# =============================================================================
# 3. Lexical Feature Extraction from URLs
# Extracting structural and statistical indicators common in phishing/malicious URLs.
# =============================================================================

def extract_url_features(url):
    parsed = urlparse(url)
    hostname = parsed.netloc
    
    features = {
        'url_length': len(url),
        'hostname_length': len(hostname),
        'dot_count': url.count('.'),
        'hyphen_count': url.count('-'),
        'slash_count': url.count('/'),
        'digit_count': sum(c.isdigit() for c in url),
        'has_https': 1 if parsed.scheme == 'https' else 0,
        'has_ip': 1 if re.search(r'\d+\.\d+\.\d+\.\d+', hostname) else 0,
        'has_suspicious_words': 1 if re.search(r'(login|verify|update|secure|account|bank|free|claim|kyc|exe|zip)', url.lower()) else 0
    }
    return pd.Series(features)

X = df['URL'].apply(extract_url_features)
y = df['Label']

print("--- Extracted URL Features Preview ---")
print(X.head())

In [ ]:
# =============================================================================
# 4. Train-Test Split, Feature Scaling & Logistic Regression Training
# =============================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Standardize numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize and train Logistic Regression model
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train_scaled, y_train)

# Predictions
y_pred = log_reg.predict(X_test_scaled)
y_prob = log_reg.predict_proba(X_test_scaled)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}")

In [ ]:
# =============================================================================
# 5. Evaluate Model & Test on Custom URLs
# =============================================================================

print("--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Malicious']))

# Plot Confusion Matrix and ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=['Legitimate', 'Malicious'], yticklabels=['Legitimate', 'Malicious'], ax=axes[0])
axes[0].set_title('Logistic Regression Confusion Matrix')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('Actual Label')

fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc_score(y_test, y_prob):.2f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1].set_title('Receiver Operating Characteristic (ROC)')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

# Test with new unseen URLs
test_urls = [
    'https://www.github.com/fastapi/fastapi',
    'http://update-bank-account-security-login.xyz/verify.php'
]
test_features = scaler.transform(pd.Series(test_urls).apply(extract_url_features))
predictions = log_reg.predict(test_features)

print("--- Sample URL Predictions ---")
for u, p in zip(test_urls, predictions):
    print(f"{u}  -->  {'Malicious' if p == 1 else 'Legitimate'}")